<a href="https://colab.research.google.com/github/madhumi2611/Airline-Ticket-Price-Prediction-using-16-ML-models/blob/main/16Models_CNN_DNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, BaggingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten, LSTM
from tensorflow.keras.callbacks import EarlyStopping

# Load dataset
file_path = "Clean_Dataset.csv"
df = pd.read_csv(file_path)

# Drop rows with missing values
df.dropna(inplace=True)

# Encode categorical features
label_encoders = {}
categorical_cols = ['airline', 'flight', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class']
for col in categorical_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

# Define features and target
X = df.drop(columns=['price'])
y = df['price']

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regression': DecisionTreeRegressor(),
    'Elastic Net': ElasticNet(),
    'XGB Regressor': XGBRegressor(),
    'K Neighbors Regressor': KNeighborsRegressor(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'SVR': SVR(),
    'Extra Trees Regressor': ExtraTreesRegressor(),
    'Random Forest Regressor': RandomForestRegressor(),
    'Bagging Regressor': BaggingRegressor(),
    'Gradient Boosting Regressor': GradientBoostingRegressor(),
    'MLP Regressor': MLPRegressor(),
    'AdaBoost Regressor': AdaBoostRegressor()
}

# Evaluate models
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    adjusted_r2 = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)

    results.append([name, mae, mse, rmse, r2, adjusted_r2])

# Deep Learning Models
# Define Deep Neural Network (DNN)
dnn = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])
dnn.compile(optimizer='adam', loss='mse')

# Train DNN
dnn.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), callbacks=[EarlyStopping(patience=5)], verbose=0)
y_pred = dnn.predict(X_test).flatten()
r2 = r2_score(y_test, y_pred)
adjusted_r2 = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
results.append(["Deep Neural Network", mean_absolute_error(y_test, y_pred), mean_squared_error(y_test, y_pred), np.sqrt(mean_squared_error(y_test, y_pred)), r2, adjusted_r2])

# Define CNN for structured data
cnn = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])
cnn.compile(optimizer='adam', loss='mse')

# Reshape input for CNN
X_train_cnn = np.expand_dims(X_train, axis=-1)
X_test_cnn = np.expand_dims(X_test, axis=-1)

# Train CNN
cnn.fit(X_train_cnn, y_train, epochs=50, batch_size=32, validation_data=(X_test_cnn, y_test), callbacks=[EarlyStopping(patience=5)], verbose=0)
y_pred = cnn.predict(X_test_cnn).flatten()
r2 = r2_score(y_test, y_pred)
adjusted_r2 = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
results.append(["Convolutional Neural Network", mean_absolute_error(y_test, y_pred), mean_squared_error(y_test, y_pred), np.sqrt(mean_squared_error(y_test, y_pred)), r2, adjusted_r2])

# Convert results to DataFrame
results_df = pd.DataFrame(results, columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2', 'Adjusted R2'])

# Display results
print(results_df.sort_values(by='R2', ascending=False))

/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1876/1876 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1876/1876 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
                           Model           MAE           MSE          RMSE  \
9        Random Forest Regressor    770.583128  4.556666e+06   2134.634875   
10             Bagging Regressor    797.763907  4.970103e+06   2229.372848   
8          Extra Trees Regressor    786.062968  4.987352e+06   2233.238002   
1       Decision Tree Regression    763.104529  7.368564e+06   2714.509811   
3                  XGB Regressor   1589.529663  8.215154e+06   2866.209082   
4          K Neighbors Regressor   1720.269074  1.294421e+07   3597.806656   
15  Convolutional Neural Network   2334.667969  1.658006e+07   4071.861859   
14           Deep Neural Network   2435.468018  1.797061e+07   4239.175156   
11   Gradient Boosting Regressor   2714.570290  2.092390e+07   4574.265210   
12                 MLP Regressor   3376.706851  3.068209e+07   5539.142029   
13            AdaBoost Regressor   4595.249654  4.186058e+07   6469.975415   
6                    